# Lab 13: Fine-Tuning a Domain-Specific Academic Assistant (LLMs)

**Course:** COMP-341L - Artificial Neural Networks Lab  
**Student:** Ali Hamza  
**Roll Number:** B23F0063AI106  
**Section:** B.S AI - Red  
**Execution Environment:** Google Colab

## Objective
Build a small, domain-specific academic assistant that explains CS concepts in a **custom teaching style**.

This notebook is designed to satisfy the lab requirements:
- Manual dataset (instruction → output) in a strict teaching style
- Model selection + justification (small model)
- Fine-tuning strategy (LoRA)
- Evaluation (base vs fine-tuned) using the same prompt
- Failure analysis (2 cases) and 1 manual improvement + re-evaluation


In [1]:
import os
from datetime import datetime

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

STUDENT_NAME = 'Ali Hamza'
STUDENT_ROLL = 'B23F0063AI106'
STUDENT_SECTION = 'B.S AI - Red'
STUDENT_FOLDER_NAME = "Ali Hamza's Lab"
USE_GOOGLE_DRIVE = True

if IN_COLAB:
    if not USE_GOOGLE_DRIVE:
        raise RuntimeError("Set USE_GOOGLE_DRIVE=True to save everything on Google Drive.")
    drive.mount("/content/drive", force_remount=True)
    BASE_DIR = f"/content/drive/MyDrive/COMP-341L/Lab 13/{STUDENT_FOLDER_NAME}"
    print("Google Drive mounted successfully.")
else:
    BASE_DIR = os.environ.get("LAB13_BASE_DIR", ".")

DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
ADAPTER_DIR = os.path.join(OUTPUTS_DIR, "lora_adapter")
ADAPTER_DIR_V2 = os.path.join(OUTPUTS_DIR, "lora_adapter_v2")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("IN_COLAB      :", IN_COLAB)
print("USE_GOOGLE_DRIVE:", USE_GOOGLE_DRIVE)
print("BASE_DIR      :", os.path.abspath(BASE_DIR))
print("DATA_DIR      :", os.path.abspath(DATA_DIR))
print("OUTPUTS_DIR   :", os.path.abspath(OUTPUTS_DIR))


Mounted at /content/drive
Google Drive mounted successfully.
IN_COLAB      : True
USE_GOOGLE_DRIVE: True
BASE_DIR      : /content/drive/MyDrive/COMP-341L/Lab 13/Ali Hamza's Lab
DATA_DIR      : /content/drive/MyDrive/COMP-341L/Lab 13/Ali Hamza's Lab/data
OUTPUTS_DIR   : /content/drive/MyDrive/COMP-341L/Lab 13/Ali Hamza's Lab/outputs


## Part 1 + Part 2 — Dataset (Manual) + Teaching Style

The dataset is stored as JSONL in the lab folder:
- `data/dataset_cs_instructions_v1.jsonl`
- `data/dataset_cs_instructions_v2.jsonl` (after manual improvement)

**Teaching style (strict):**
1. Step-by-step explanation
2. A real-life analogy
3. A tiny example (1–3 lines)
4. A common mistake / misconception
5. A one-line recap

The goal of fine-tuning is not to “teach new facts” to the model, but to **make outputs clearer, more consistent, and less likely to guess**.


In [2]:
import json
from pathlib import Path

v1_path = Path(DATA_DIR) / "dataset_cs_instructions_v1.jsonl"
v2_path = Path(DATA_DIR) / "dataset_cs_instructions_v2.jsonl"

# If you opened the notebook directly from the Drive folder, the dataset should already exist.
# This fallback is only to keep the notebook runnable end-to-end.
if not v1_path.exists():
    raise FileNotFoundError(
        f"Missing dataset file: {v1_path}. Make sure you copied the lab folder to Google Drive."
    )

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

data_v1 = read_jsonl(v1_path)
data_v2 = read_jsonl(v2_path) if v2_path.exists() else []

print("Loaded v1 samples:", len(data_v1))
print("Loaded v2 samples:", len(data_v2))
print("Example v1 item keys:", data_v1[0].keys())


Loaded v1 samples: 26
Loaded v2 samples: 5
Example v1 item keys: dict_keys(['instruction', 'output'])


## Part 3 — Model Selection (Small LLM)

**Chosen model:** `distilgpt2`

**Why this model (reasoning):**
- Small enough to fine-tune quickly on Colab
- Good for demonstrating fine-tuning behavior with limited resources
- Still produces coherent text for short explanations

**Limitations (important):**
- Weaker factual reliability than larger models
- More likely to hallucinate if the prompt is vague
- Limited context window and reasoning depth


## Part 4 — Fine-Tuning Strategy (LoRA)

We use **LoRA (Low-Rank Adaptation)** to train only a small number of additional parameters (adapters),
instead of updating all model weights.

**Why LoRA is suitable here:**
- Faster training on limited hardware
- Lower memory usage
- Keeps base model intact; we can compare base vs adapter easily


In [3]:
import sys
import subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

# Pin versions to avoid common Colab dependency mismatches (peft/transformers/huggingface_hub).
pip_install(
    [
        "transformers==4.41.2",
        "datasets==2.19.1",
        "peft==0.11.1",
        "accelerate==0.31.0",
        "huggingface_hub==0.23.4",
        "safetensors",
    ]
)

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

device: cuda


In [4]:
BASE_MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(instruction: str, output: str) -> str:
    return f"### Instruction:\n{instruction}\n\n### Response:\n{output}\n"

def to_dataset(rows):
    texts = [format_example(r["instruction"], r["output"]) for r in rows]
    return Dataset.from_dict({"text": texts})

ds_v1 = to_dataset(data_v1)
print(ds_v1)
print(ds_v1[0]["text"][:250])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Dataset({
    features: ['text'],
    num_rows: 26
})
### Instruction:
Explain recursion using a real-life analogy for a beginner.

### Response:
Step-by-step:
1) Idea: Recursion is a function solving a problem by calling itself on a smaller version.
2) Two key parts: (a) a base case that stops, and (b)


In [5]:
def tokenize_batch(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized_v1 = ds_v1.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map:   0%|          | 0/26 [00:00<?, ? examples/s]

In [6]:
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)
base_model.resize_token_embeddings(len(tokenizer))
base_model.to(device)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 405,504 || all params: 82,318,080 || trainable%: 0.4926


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:1119: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## Fine-tune (v1 dataset)

Notes:
- This is a tiny dataset; the goal is to learn *style and format*, not broad knowledge.
- Keep epochs small to avoid overfitting / repetition.


In [7]:
training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUTS_DIR, "trainer_runs_v1"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=8,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_v1,
    data_collator=data_collator,
)

trainer.train()
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved LoRA adapter to:", ADAPTER_DIR)


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved LoRA adapter to: /content/drive/MyDrive/COMP-341L/Lab 13/Ali Hamza's Lab/outputs/lora_adapter


## Part 5 — Evaluation (Base vs Fine-Tuned)

Test prompt (fixed): **Explain dynamic programming**


In [8]:
import textwrap

def generate_text(model_, prompt: str, max_new_tokens: int = 220):
    model_.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = model_.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

eval_instruction = "Explain dynamic programming in a step-by-step, analogy-based teaching style."
eval_prompt = f"### Instruction:\n{eval_instruction}\n\n### Response:\n"

base_eval_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(device)
tuned_eval_model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(device),
    ADAPTER_DIR,
).to(device)

base_out = generate_text(base_eval_model, eval_prompt)
tuned_out = generate_text(tuned_eval_model, eval_prompt)

print("\n" + "=" * 40 + "\nBASE MODEL OUTPUT\n" + "=" * 40)
print(base_out)
print("\n" + "=" * 40 + "\nFINE-TUNED OUTPUT (v1)\n" + "=" * 40)
print(tuned_out)

(Path(OUTPUTS_DIR) / "eval_base.txt").write_text(base_out, encoding="utf-8")
(Path(OUTPUTS_DIR) / "eval_finetuned_v1.txt").write_text(tuned_out, encoding="utf-8")
print("\nSaved eval outputs to outputs/: eval_base.txt, eval_finetuned_v1.txt")



BASE MODEL OUTPUT
### Instruction:
Explain dynamic programming in a step-by-step, analogy-based teaching style.

### Response:
This is a great introduction to the following topics, which are covered in this tutorial.
Note: This is not a complete tutorial.
### The lesson is a part of the course, so please read the full course description.

FINE-TUNED OUTPUT (v1)
### Instruction:
Explain dynamic programming in a step-by-step, analogy-based teaching style.

### Response:
This post is part of a series that focuses on the language of programming in a way that is not currently possible in the language of programming. It has been translated into English and Spanish.
The most recent version of this article was translated into Spanish.
There are three main problems with the language of programming in a way that is not currently possible in the language of programming.
First, it is not possible to use a language of programming in a way that is not currently possible in the language of programmi

## Part 6 — Failure Analysis (at least 2 cases)

Below are two **intentionally chosen** prompts that often expose weak behavior in small LLMs:
1) **Precision failure:** Ask for a strict, technical definition (model may get details wrong).
2) **Overconfidence/hallucination:** Ask for details the model may “guess”.


In [9]:
failure_prompts = [
    "Explain dynamic programming and prove why it always gives the optimal answer.",
    "Explain the exact time complexity of the best-known algorithm for the traveling salesman problem and cite the year it was discovered.",
]

def run_failures(tag: str, model_):
    outs = {}
    for i, instr in enumerate(failure_prompts, start=1):
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"
        out = generate_text(model_, prompt, max_new_tokens=220)
        outs[f"failure_{i}"] = out
        print("\n" + "-" * 30)
        print(f"{tag} | failure_{i} prompt:\n{instr}\n")
        print(out)
    return outs

base_fail = run_failures("BASE", base_eval_model)
tuned_fail_v1 = run_failures("TUNED_V1", tuned_eval_model)

(Path(OUTPUTS_DIR) / "failures_base.json").write_text(json.dumps(base_fail, indent=2), encoding="utf-8")
(Path(OUTPUTS_DIR) / "failures_finetuned_v1.json").write_text(json.dumps(tuned_fail_v1, indent=2), encoding="utf-8")
print("\nSaved failure outputs to outputs/: failures_base.json, failures_finetuned_v1.json")



------------------------------
BASE | failure_1 prompt:
Explain dynamic programming and prove why it always gives the optimal answer.

### Instruction:
Explain dynamic programming and prove why it always gives the optimal answer.

### Response:
To help answer questions about programming, you should read the following section.
This section is for those who have used the following tools:
http://www.youtube.com/watch?v=xW3ZHbEZrK
http://www.youtube.com/watch?v=0vw5qFq2F0
http://www.youtube.com/watch?v=jZwk4UvS8k
http://www.youtube.com/watch?v=nC1KmZkU
http://www.youtube.com/watch?v=ySgVxG5rX0
http://www.youtube.com/watch?v=z9dHkWQ1g
http://www.youtube.com/watch?v=1JwfPdX0
http://www.youtube.com/watch?v=8UwFq2F0
http://www.youtube.com/watch?v=0j3

------------------------------
BASE | failure_2 prompt:
Explain the exact time complexity of the best-known algorithm for the traveling salesman problem and cite the year it was discovered.

### Instruction:
Explain the exact time complexity of 

## Part 7 — Manual Improvement (Fix ONE failure)

Manual improvement strategy used here:
- Add higher-quality examples that **teach the model how to respond when unsure** and how to define DP state clearly.
- Train a second adapter on `dataset_cs_instructions_v2.jsonl`.

Then we re-run the same evaluation prompt and compare.


In [10]:
if not v2_path.exists():
    raise FileNotFoundError(
        f"Missing improved dataset file: {v2_path}. Make sure it exists in your lab folder."
    )

ds_v2 = to_dataset(read_jsonl(v2_path))
tokenized_v2 = ds_v2.map(tokenize_batch, batched=True, remove_columns=["text"])

base_model_v2 = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)
base_model_v2.resize_token_embeddings(len(tokenizer))
base_model_v2.to(device)
model_v2 = get_peft_model(base_model_v2, lora_config)

training_args_v2 = TrainingArguments(
    output_dir=os.path.join(OUTPUTS_DIR, "trainer_runs_v2"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=6,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=tokenized_v2,
    data_collator=data_collator,
)

trainer_v2.train()
model_v2.save_pretrained(ADAPTER_DIR_V2)
tokenizer.save_pretrained(ADAPTER_DIR_V2)
print("Saved LoRA adapter v2 to:", ADAPTER_DIR_V2)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved LoRA adapter v2 to: /content/drive/MyDrive/COMP-341L/Lab 13/Ali Hamza's Lab/outputs/lora_adapter_v2


In [11]:
tuned_eval_model_v2 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(device),
    ADAPTER_DIR_V2,
).to(device)

tuned_out_v2 = generate_text(tuned_eval_model_v2, eval_prompt)
print("\n" + "=" * 40 + "\nFINE-TUNED OUTPUT (v2)\n" + "=" * 40)
print(tuned_out_v2)

(Path(OUTPUTS_DIR) / "eval_finetuned_v2.txt").write_text(tuned_out_v2, encoding="utf-8")

tuned_fail_v2 = run_failures("TUNED_V2", tuned_eval_model_v2)
(Path(OUTPUTS_DIR) / "failures_finetuned_v2.json").write_text(json.dumps(tuned_fail_v2, indent=2), encoding="utf-8")
print("\nSaved outputs/: eval_finetuned_v2.txt, failures_finetuned_v2.json")



FINE-TUNED OUTPUT (v2)
### Instruction:
Explain dynamic programming in a step-by-step, analogy-based teaching style.

### Response:
"I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going to say that I am going

--------------------------

## Reflection Report Export (Markdown + HTML)

This section writes a structured report file to your Drive folder.
It pulls in the saved model outputs to ensure the report contains **before/after evidence**.


In [12]:
from pathlib import Path
import html

report_md_path = Path(BASE_DIR) / "Lab_Report_13.md"
report_html_path = Path(BASE_DIR) / "Lab_Report_13.html"

def safe_read(path: Path):
    return path.read_text(encoding="utf-8") if path.exists() else "(missing)"

eval_base = safe_read(Path(OUTPUTS_DIR) / "eval_base.txt")
eval_v1 = safe_read(Path(OUTPUTS_DIR) / "eval_finetuned_v1.txt")
eval_v2 = safe_read(Path(OUTPUTS_DIR) / "eval_finetuned_v2.txt")
failures_base = safe_read(Path(OUTPUTS_DIR) / "failures_base.json")
failures_v1 = safe_read(Path(OUTPUTS_DIR) / "failures_finetuned_v1.json")
failures_v2 = safe_read(Path(OUTPUTS_DIR) / "failures_finetuned_v2.json")

today = datetime.now().strftime("%B %d, %Y")

report_md = f\"\"\"# Lab 13 — Fine-Tuning a Domain-Specific Academic Assistant (LLMs)

**Student:** Ali Hamza  
**Roll Number:** B23F0063AI106  
**Section:** B.S AI - Red  
**Date:** {today}

## Part 1 — Dataset Creation (Manual)
- v1 dataset: `data/dataset_cs_instructions_v1.jsonl` ({len(data_v1)} samples)
- v2 dataset: `data/dataset_cs_instructions_v2.jsonl` ({len(read_jsonl(v2_path))} samples)
- Format: JSONL with fields `instruction`, `output`
- Manual constraint: Explanations written in a single consistent teaching style

## Part 2 — Teaching Style Definition
**Style rules used in every output:**
1) Step-by-step explanation  
2) Analogy  
3) Tiny example  
4) Common mistake  
5) One-line recap

## Part 3 — Model Selection
**Selected:** `distilgpt2`
- Advantages: small, fast to fine-tune, works on Colab
- Limitations: more hallucination risk, weaker reasoning depth, limited context

## Part 4 — Fine-Tuning Strategy
**Selected:** LoRA (PEFT)
- Why: trains few parameters, faster + memory efficient
- Effect: base weights frozen; only low-rank adapter matrices updated

## Part 5 — Evaluation (Same Prompt)
**Prompt:** Explain dynamic programming

### Base model output
```text
{eval_base}
```

### Fine-tuned output (v1)
```text
{eval_v1}
```

### Fine-tuned output (v2, after manual improvement)
```text
{eval_v2}
```

**Critical comparison (write your analysis in your own words):**
- Clarity: …
- Hallucination: …
- Style consistency: …

## Part 6 — Failure Analysis (2 cases)
**Base failures (raw):**
```json
{failures_base}
```

**Fine-tuned v1 failures (raw):**
```json
{failures_v1}
```

**Fine-tuned v2 failures (raw):**
```json
{failures_v2}
```

**Your analysis (why it failed; data vs model vs training):**
- Failure 1: …
- Failure 2: …

## Part 7 — Manual Improvement (Fix one failure)
- Change made: switched to v2 dataset with improved examples (DP state + uncertainty-aware answers)
- Observed effect: …

## Reflection (Key Insights)
- What the model learned: …
- Where it still failed: …
- What you would try next (more data / better prompts / eval metrics): …
\"\"\"

report_md_path.write_text(dedent(report_md).strip() + \"\\n\", encoding=\"utf-8\")

# Minimal HTML export (keeps it simple and offline-friendly)
html_body = \"<pre>\" + html.escape(report_md_path.read_text(encoding=\"utf-8\")) + \"</pre>\"
report_html_path.write_text(
    \"<html><head><meta charset='utf-8'><title>Lab Report 13</title></head><body>\" + html_body + \"</body></html>\",
    encoding=\"utf-8\",
)

print(\"Wrote:\", report_md_path)
print(\"Wrote:\", report_html_path)


SyntaxError: unexpected character after line continuation character (4157075020.py, line 19)